In [ ]:
import pandas as pd


# ============================
# 1. Load the data
# ============================
holdout_df = pd.read_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\raw\holdout.csv')
train_df = pd.read_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\raw\train.csv')
eval_df = pd.read_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\raw\eval.csv')
metros= pd.read_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\raw\usmetros.csv')

pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.max_rows', None) # show all rows

In [ ]:
print(train_df.shape)
print(eval_df.shape)

In [ ]:
train_df['city_full'].value_counts().head()

<details>
  <summary>Map cities to Lat/Long</summary>
  
  * The goal is to use Latitude and longitude...
</details>


In [ ]:
# =========================
# 2. Fix city name mismatches
# =========================

city_mapping = {
    'New York-Newark-Jersey City': 'New York-Newark-Jersey City, NY-NJ',
    'Los Angeles-Long Beach-Anaheim': 'Los Angeles-Long Beach-Anaheim, CA',
    'Chicago-Naperville-Elgin': 'Chicago-Naperville-Elgin, IL-IN',
    'Dallas-Fort Worth-Arlington': 'Dallas-Fort Worth-Arlington, TX',
    'Houston-Pasadena-The Woodlands': 'Houston-Pasadena-The Woodlands, TX',
    'Atlanta-Sandy Springs-Roswell': 'Atlanta-Sandy Springs-Roswell, GA',
    'Washington-Arlington-Alexandria': 'Washington-Arlington-Alexandria, DC-VA-MD-WV',
    'Philadelphia-Camden-Wilmington': 'Philadelphia-Camden-Wilmington, PA-NJ-DE-MD',
    'Miami-Fort Lauderdale-West Palm Beach': 'Miami-Fort Lauderdale-West Palm Beach, FL',
    'Phoenix-Mesa-Chandler': 'Phoenix-Mesa-Chandler, AZ',
    'Boston-Cambridge-Newton': 'Boston-Cambridge-Newton, MA-NH',
    'Riverside-San Bernardino-Ontario': 'Riverside-San Bernardino-Ontario, CA',
    'San Francisco-Oakland-Fremont': 'San Francisco-Oakland-Fremont, CA',
    'Detroit-Warren-Dearborn': 'Detroit-Warren-Dearborn, MI',
    'Seattle-Tacoma-Bellevue': 'Seattle-Tacoma-Bellevue, WA',
    'Minneapolis-St. Paul-Bloomington': 'Minneapolis-St. Paul-Bloomington, MN-WI',
    'Tampa-St. Petersburg-Clearwater': 'Tampa-St. Petersburg-Clearwater, FL',
    'San Diego-Chula Vista-Carlsbad': 'San Diego-Chula Vista-Carlsbad, CA',
    'Denver-Aurora-Centennial': 'Denver-Aurora-Centennial, CO',
    'Baltimore-Columbia-Towson': 'Baltimore-Columbia-Towson, MD',
    'Orlando-Kissimmee-Sanford': 'Orlando-Kissimmee-Sanford, FL',
    'Charlotte-Concord-Gastonia': 'Charlotte-Concord-Gastonia, NC-SC',
    'St. Louis': 'St. Louis, MO-IL',
    'San Antonio-New Braunfels': 'San Antonio-New Braunfels, TX',
    'Portland-Vancouver-Hillsboro': 'Portland-Vancouver-Hillsboro, OR-WA',
    'Austin-Round Rock-San Marcos': 'Austin-Round Rock-San Marcos, TX',
    'Pittsburgh': 'Pittsburgh, PA',
    'Sacramento-Roseville-Folsom': 'Sacramento-Roseville-Folsom, CA',
    'Las Vegas-Henderson-North Las Vegas': 'Las Vegas-Henderson-North Las Vegas, NV',
    'Cincinnati': 'Cincinnati, OH-KY-IN',
    'Atlanta-Sandy Springs-Roswell': 'Atlanta-Sandy Springs-Roswell, GA',
    'Las Vegas-Henderson-North Las Vegas': 'Las Vegas-Henderson-North Las Vegas, NV',
    'San Francisco-Oakland-Fremont': 'San Francisco-Oakland-Fremont, CA',
    'Austin-Round Rock-San Marcos': 'Austin-Round Rock-San Marcos, TX',
    'Washington-Arlington-Alexandria': 'Washington-Arlington-Alexandria, DC-VA-MD-WV',
    'Denver-Aurora-Centennial': 'Denver-Aurora-Centennial, CO',
    'Miami-Fort Lauderdale-West Palm Beach': 'Miami-Fort Lauderdale-West Palm Beach, FL',
    'Houston-Pasadena-The Woodlands': 'Houston-Pasadena-The Woodlands, TX',
    
    # These were previously mapped to simplified names, but the metros file needs the full string
    'Atlanta-Sandy Springs-Alpharetta': 'Atlanta-Sandy Springs-Roswell, GA',
    'San Francisco-Oakland-Berkeley': 'San Francisco-Oakland-Fremont, CA',
    'Austin-Round Rock-Georgetown': 'Austin-Round Rock-San Marcos, TX',
    'Houston-The Woodlands-Sugar Land': 'Houston-Pasadena-The Woodlands, TX',
    'Denver-Aurora-Lakewood': 'Denver-Aurora-Centennial, CO',
    'DC_Metro': 'Washington-Arlington-Alexandria, DC-VA-MD-WV',
    'Las Vegas-Henderson-Paradise': 'Las Vegas-Henderson-North Las Vegas, NV',
    'Miami-Fort Lauderdale-Pompano Beach': 'Miami-Fort Lauderdale-West Palm Beach, FL'
}


In [ ]:
print(metros.columns)

In [ ]:
def clean_and_merge(df: pd.DataFrame) -> pd.DataFrame:
    """Apply city name fixes, merge lat/lng from metros, drop dup col."""

    # fix city name mismatches
    df['city_full'] = df['city_full'].replace(city_mapping)

    # merge lat/lng from metros
    df = df.merge(metros[['metro_full', 'lat', 'lng']],
                  how='left',
                  left_on='city_full',
                  right_on='metro_full'
    )

    # drop duplicate lat/lng columns

    df = df.drop(columns=['metro_full'])

    # log any cities that still didn't match
    missing = df[df['lat'].isnull()]['city_full'].unique()
    if len(missing) > 0:
        print(f"Warning: the following cities didn't match any in the metros file: {missing}")
    else:
        print("All cities matched successfully.")
    return df




In [ ]:
#===========================
# 3. Apply cleaning + merge to both train and eval df
#===========================

train_df = clean_and_merge(train_df)
eval_df = clean_and_merge(eval_df)
holdout_df = clean_and_merge(holdout_df)

In [ ]:
train_df.head(1)

**Clean Duplicates**

In [ ]:
print(train_df.shape)

duplicated_rows = train_df[train_df.duplicated()].shape[0]
print(f"Number of duplicated rows in train_df: {duplicated_rows}")

duplicated_rows = train_df[train_df.duplicated(subset=train_df.columns.difference(['date', 'year']))].shape[0]
print(f"Number of duplicated rows in train_df (ignoring date/year): {duplicated_rows}")



In [ ]:
print(eval_df.shape)

duplicated_rows = eval_df[eval_df.duplicated()].shape[0]
print(f"Number of duplicated rows in eval_df: {duplicated_rows}")

duplicated_rows = eval_df[eval_df.duplicated(subset = eval_df.columns.difference(['date','year']))].shape[0]
print(f"Number of duplicated rows in eval_df (ignoring date/year): {duplicated_rows}")

In [ ]:
print(holdout_df.shape)
duplicated_rows = holdout_df[holdout_df.duplicated()].shape[0]
print(f"Number of duplicated rows in holdout_df: {duplicated_rows}")

duplicated_rows = holdout_df[holdout_df.duplicated(subset=holdout_df.columns.difference(['date','year']))].shape[0]
print(f"Number of duplicated rows in holdout_df (ignoring date/year): {duplicated_rows}")

In [ ]:
# Delete the duplicates from training set
train_df = train_df.drop_duplicates(subset=train_df.columns.difference(['date', 'year']),keep = False)

print(train_df.shape)

duplicated_rows = train_df[train_df.duplicated()].shape[0]
print(f"Number of duplicated rows in train_df: {duplicated_rows}")

duplicated_rows = train_df[train_df.duplicated(subset=train_df.columns.difference(['date', 'year']))].shape[0]
print(f"Number of duplicated rows in train_df (ignoring date/year): {duplicated_rows}")

In [ ]:
# Delete the duplicates from evaluation set
eval_df = eval_df.drop_duplicates(subset=eval_df.columns.difference(['date', 'year']),keep = False)

print(eval_df.shape)

duplicated_rows = eval_df[eval_df.duplicated()].shape[0]
print(f"Number of duplicated rows in eval_df: {duplicated_rows}")

duplicated_rows = eval_df[eval_df.duplicated(subset=eval_df.columns.difference(['date', 'year']))].shape[0]
print(f"Number of duplicated rows in eval_df (ignoring date/year): {duplicated_rows}")

In [ ]:
# Delete the duplicates from holdout set
holdout_df = holdout_df.drop_duplicates(subset=holdout_df.columns.difference(['date', 'year']),keep = False)
print(holdout_df.shape)

duplicated_rows = holdout_df[holdout_df.duplicated()].shape[0]
print(f"Number of duplicated rows in holdout_df: {duplicated_rows}")

duplicated_rows = holdout_df[holdout_df.duplicated(subset=holdout_df.columns.difference(['date', 'year']))].shape[0]
print(f"Number of duplicated rows in holdout_df (ignoring date/year): {duplicated_rows}")

**Clean OutLiers**

In [ ]:
train_df['median_list_price'].describe()

In [ ]:
import plotly.express as px

fig = px.violin(train_df, y='median_list_price',box = True, hover_name = 'median_list_price')
fig.update_layout(title = 'Distribution of median_list_price')
fig.show()


In [ ]:
top_1_percent = train_df.nlargest(int(len(train_df) * 0.01), 'median_list_price')
print(top_1_percent.shape)
top_1_percent.head(10)

In [ ]:
print(top_1_percent['median_list_price'].value_counts().sort_index(ascending=False))


- investigate if median_list_price are independent in destinct regions(for example if DC has different median_list_price)

- Drop outliers to keep things realistic and clean

In [ ]:
# Clean outliers above 19M in both the train and eval datasets
train_df = train_df[train_df['median_list_price'] <= 19_000_000].copy()
eval_df = eval_df[eval_df['median_list_price'] <= 19_000_000].copy()
holdout_df = holdout_df[holdout_df['median_list_price'] <= 19_000_000].copy()

In [ ]:
import plotly.express as px
fig = px.violin(train_df, y = 'median_list_price', box = True, hover_name ='median_list_price')
fig.update_layout(title = 'Distribution of median_list_price (after outlier removal)')
fig.show()

In [ ]:
top_1_percent = train_df.nlargest(int(len(train_df)*0.01), 'median_list_price')
print(top_1_percent['median_list_price'].value_counts().sort_index(ascending=False))

In [ ]:
#==================================================
# 4. Save cleaned datasets
#==================================================

train_df.to_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\processed\train_cleaned.csv', index=False)
eval_df.to_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\processed\eval_cleaned.csv', index=False)
holdout_df.to_csv(r'E:\End-to-end-machine-learning-project-work-flow\data\processed\holdout_cleaned.csv', index=False)

print("Cleaning complete.")

### Housing Prices exploration ###

In [ ]:
df = train_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import colorsys
           

# Boxplot of house prices for the cities, ordered by median price.

top_cities = df["city"].value_counts().head(30).index.tolist()
df_city    = df[df["city"].isin(top_cities)]
city_order = (
    df_city.groupby("city")["price"]
           .median()
           .sort_values(ascending=False)
           .index
)

base = sns.color_palette("rocket_r", len(city_order))

def lighten(color, amt=.4):
    h, l, s = colorsys.rgb_to_hls(*color)
    return colorsys.hls_to_rgb(h, min(1, l + amt*(1-l)), s)

palette = [
    lighten(c, .45) if i >= len(base) - 8 else c
    for i, c in enumerate(base)
]

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(14, 6))

sns.boxplot(
    x="city", y="price", data=df_city,
    order=city_order, palette=palette,
    showfliers=False, linewidth=1.2, ax=ax
)

medians = df_city.groupby("city")["price"].median().loc[city_order]
for tick, median in enumerate(medians):
    ax.scatter(tick, median, color="white", edgecolor="black", zorder=5, s=40)

ax.set_xlabel("")
ax.set_ylabel("Price ($)")
ax.tick_params(axis="x", rotation=55)
sns.despine(trim=True)
ax.grid(False)
ax.ticklabel_format(axis="y", style="plain")


In [ ]:
# Distribution of house prices across the dataset.

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(8, 6))
sns.histplot(df["price"].dropna(), bins=60, kde=True, color= sns.color_palette('rocket_r',1)[0], ax=ax)
median_price = df['price'].median()
ax.axvline(median_price , color = 'black', lw = 1.2, ls='--')
ax.set_xlabel("Price ($)")
ax.set_ylabel("Count")
sns.despine(trim=True)
ax.grid(False)
ax.ticklabel_format(axis="x", style="plain")
plt.show()
